# K-Nearest Neighbors (KNN) para la Detección de Malware en Android

Este notebook implementa un modelo **K-Nearest Neighbors (KNN)** para la clasificación binaria de aplicaciones Android como benignas o maliciosas utilizando el dataset **NATICUSdroid**.

La metodología experimental desarrollada incluye:

- carga del conjunto de datos
- preparación de variables
- partición entrenamiento/prueba
- normalización de características
- optimización de hiperparámetros mediante GridSearchCV
- validación cruzada estratificada
- evaluación del modelo con métricas de clasificación
- visualización de matriz de confusión
- análisis mediante curva ROC

El objetivo es construir un modelo no paramétrico para comparar su desempeño frente a otros enfoques de aprendizaje automático.

## 1. Importación de librerías

En esta sección se importan las librerías necesarias para:

- manipulación de datos
- visualización gráfica
- entrenamiento del modelo
- optimización de hiperparámetros
- evaluación del desempeño del clasificador

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

## 2. Carga del conjunto de datos

Se carga el dataset NATICUSdroid desde el repositorio del proyecto.

Este conjunto contiene variables binarias asociadas a permisos solicitados por aplicaciones Android, así como la variable objetivo correspondiente a la clasificación de benigno o malware.

In [ ]:
df = pd.read_csv("../data/raw/data.csv")

print("Dataset shape:", df.shape)
df.head()

## 3. Preparación de los datos

Se separa la variable objetivo (`Result`) de las variables predictoras.

Posteriormente, se realiza una partición estratificada entre entrenamiento y prueba para conservar la distribución de clases original.

In [ ]:
X = df.drop(columns=["Result"])
y = df["Result"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

## 4. Configuración del modelo y optimización de hiperparámetros

Se construye un pipeline compuesto por:

- StandardScaler, para normalización de variables
- K-Nearest Neighbors como clasificador no paramétrico

Se realiza optimización de hiperparámetros mediante GridSearchCV con validación cruzada estratificada de 5 particiones.

La métrica objetivo seleccionada es F1-score.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best CV F1-score:")
print(grid.best_score_)

## 5. Evaluación del modelo final

El mejor modelo encontrado se evalúa utilizando el conjunto de prueba independiente.

Se calculan métricas estándar de clasificación para evaluar desempeño general y discriminativo.

In [ ]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

specificity = tn / (tn + fp)

print("RESULTADOS DE EVALUACIÓN DEL MODELO")
print("====================================\n")

print(f"Accuracy (Exactitud global del modelo): {accuracy_score(y_test, y_pred):.4f}")
print("→ Indica el porcentaje total de predicciones correctas.\n")

print(f"Precision (Precisión de detección de malware): {precision_score(y_test, y_pred):.4f}")
print("→ Mide qué tan confiables son las predicciones positivas del modelo.\n")

print(f"Recall / Sensibilidad (Capacidad de detectar malware): {recall_score(y_test, y_pred):.4f}")
print("→ Indica qué proporción del malware real fue correctamente detectado.\n")

print(f"F1-score (Balance entre precisión y sensibilidad): {f1_score(y_test, y_pred):.4f}")
print("→ Resume el equilibrio entre falsos positivos y falsos negativos.\n")

print(f"Specificity (Capacidad de identificar aplicaciones benignas): {specificity:.4f}")
print("→ Mide qué tan bien el modelo reconoce aplicaciones no maliciosas.\n")

print(f"ROC-AUC (Capacidad discriminativa global): {roc_auc_score(y_test, y_prob):.4f}")
print("→ Evalúa qué tan bien el modelo separa ambas clases en distintos umbrales.")













## 6. Matriz de confusión

La matriz de confusión permite analizar detalladamente el comportamiento del clasificador sobre el conjunto de prueba.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens"
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("KNN - Confusion Matrix")

plt.show()

## 7. Curva ROC

Se evalúa la capacidad discriminativa global del modelo mediante la curva ROC y el valor ROC-AUC.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))

plt.plot(fpr, tpr, label="KNN")
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()

## 8. Conclusiones

El modelo KNN permite establecer una línea base no paramétrica para la detección de malware Android.

Su desempeño permitirá comparar el efecto de modelos basados en proximidad frente a enfoques lineales y ensambles.